# Advanced Analytics & Reporting Tutorial

This tutorial demonstrates front-office risk reports and publication-quality visualisation: VaR/CVaR summary, combined RiskReport, Greeks surface, PnL by scenario, and figure export.

**Topics covered:**
- VaR summary report and combined RiskReport (Greeks, VaR, attribution, scenarios)
- Greeks surface plotter (2D heatmap)
- PnL by scenario and attribution bar plots
- Report styling and batch export (PNG + PDF)

In [ ]:
# Setup
import sys
sys.path.insert(0, '../../..')

import numpy as np
from pathlib import Path

np.random.seed(42)
print("Setup complete.")

## 1. VaR summary report

Wrap a VaR result in a summary report with optional factor breakdown; use to_console, to_dicts, to_csv.

In [ ]:
from src.risk.var import historical_var, VarConfig
from src.risk.reporting.var_summary import build_var_summary_report, VarBreakdownRow

# Simulate daily P&L and compute historical VaR
pnl_series = np.random.randn(252) * 10_000
config = VarConfig(confidence=0.99, horizon_days=1, method="historical")
var_result = historical_var(pnl_series, config)

# Build summary report (optional breakdown for parametric)
var_summary = build_var_summary_report(var_result)
print(var_summary.to_console())
print("\n--- to_dicts (nested) ---")
print(var_summary.to_dicts())

## 2. Combined risk report

Assemble Greeks summary, VaR summary, scenario report, and attribution report into a single RiskReport.

In [ ]:
from src.risk.reporting.risk_report import RiskReport
from src.risk.sensitivities.aggregation import aggregate_sensitivities
from src.risk.sensitivities.result import SensitivitiesReport, SensitivityRow, SensitivityKey
from src.risk.reporting.scenario_report import ScenarioReport, ScenarioRow
from src.marketdata.core.ids import MarketId

# Minimal Greeks summary from synthetic sensitivities
mid = MarketId("FX", "SPOT", "EURUSD")
sens_report = SensitivitiesReport(rows=[
    SensitivityRow(key=SensitivityKey(greek="delta", market_id=mid), value=50_000.0, method="analytic"),
])
greeks_summary = aggregate_sensitivities(sens_report, include_per_market_id=True)

# Minimal scenario report
scenario_report = ScenarioReport(rows=[
    ScenarioRow(scenario="BASE", pv=1_000_000.0, pnl=0.0),
    ScenarioRow(scenario="spot_up_1pct", pv=1_005_000.0, pnl=5_000.0),
    ScenarioRow(scenario="vol_up", pv=998_000.0, pnl=-2_000.0),
], base_scenario="BASE")

report = RiskReport(
    greeks_summary=greeks_summary,
    var_summary=var_summary,
    scenario_report=scenario_report,
)
print(report.to_console())

## 3. Greeks surface (2D heatmap)

Plot a greek (e.g. delta) as a function of expiry and strike. Data can come from an FD pricer sweep or from SensitivitiesReport when keys have expiry/strike.

In [ ]:
from src.core.reporting.plots.risk import plot_greeks_surface

# Dummy grid: expiries x strikes
expiries = np.array([0.25, 0.5, 1.0])
strikes = np.array([0.95, 1.0, 1.05])
delta_grid = np.array([
    [0.3, 0.5, 0.7],
    [0.35, 0.55, 0.75],
    [0.4, 0.6, 0.8],
])

fig = plot_greeks_surface(expiries, strikes, delta_grid, greek_name="delta")
fig

## 4. PnL by scenario and attribution bars

Bar charts for scenario PnL and factor attribution.

In [ ]:
from src.core.reporting.plots.risk import plot_pnl_by_scenario, plot_attribution_bars
from src.risk.attribution.report import AttributionReport, AttributionRow

fig_pnl = plot_pnl_by_scenario(scenario_report, title="Stress PnL")

# Attribution bars (minimal example)
attr_report = AttributionReport(rows=[
    AttributionRow(scenario="spot_up_1pct", pv_base=1_000_000.0, pv_scn=1_005_000.0, pnl=5_000.0,
                  contributions={"delta:FX.SPOT.EURUSD": 4_800.0}, predicted_pnl=4_800.0, residual=200.0, rel_error=0.04),
], base_scenario_name="BASE")
fig_attr = plot_attribution_bars(attr_report, title="PnL attribution: spot_up_1pct")
fig_attr

## 5. Export figures for reports

Use PlotConfig (PNG + PDF per figure) or save_report_figures for batch export.

In [ ]:
from src.core.reporting.plots.utils import PlotConfig, render_fig, save_report_figures

out_dir = Path("outputs/analytics_reports")
cfg = PlotConfig(save=True, show=False, out_dir=out_dir, save_pdf=True)

# Single figure
render_fig(fig, cfg=cfg, filename="greeks_delta_surface.png")

# Batch: report_01_vol_surface.png/pdf, report_02_greeks_delta.png/pdf, ...
figures = [(fig, "greeks_delta"), (fig_pnl, "pnl_by_scenario"), (fig_attr, "attribution_bars")]
saved = save_report_figures(figures, out_dir, prefix="risk", dpi=160)
print("Saved:", saved)

## Summary

- **VarSummaryReport** wraps VaR result with to_console / to_dicts / to_csv.
- **RiskReport** combines optional Greeks, VaR, attribution, and scenario sections.
- **plot_greeks_surface** plots a greek over (expiry, strike); use **greek_grid_from_sensitivities** when sensitivities have expiry/strike.
- **plot_pnl_by_scenario** and **plot_attribution_bars** produce publication-quality bar charts.
- **PlotConfig.save_pdf** and **save_report_figures** export PNG + PDF for reports.